# Week 2 — Pretrained Water Segmentation Using Multispectral and Optical Data

**Goal:** fine-tune an ImageNet-pretrained segmentation model (ResNet34 encoder + U-Net decoder, via
`segmentation-models-pytorch`) on the 12-channel water dataset and compare it with the Week 1 U-Net trained from scratch.

All reusable code lives in `src/` (preprocessing, data, model, losses, metrics, training engine, analysis, reporting);
this notebook only **explains, runs and displays**. The same pipeline can be run from the terminal with
`python scripts/run_week2.py`.

Fairness of the comparison: same Week 1 split (`config/split_v2.csv`), same train-derived normalisation
(`config/stats.json`), same metrics/threshold and the same seeds (42, 123, 2026).
A **random-init control** (identical architecture, no ImageNet weights) is trained as well, so the effect of
*pretraining* can be separated from the effect of the *larger architecture*.

## 1. Setup

In [ ]:
%pip install -q "segmentation-models-pytorch==0.5.0" tifffile

import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/just0neball/Water-Segmentation-using-Multispectral-and-Optical.git"

here = Path.cwd()
ROOT = next((p for p in (here, here.parent, here / "week2", here.parent / "week2")
             if (p / "src" / "pipeline.py").exists()), None)
if ROOT is None:                                   # e.g. a fresh Kaggle session
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/repo"], check=True)
    ROOT = Path("/kaggle/working/repo/week2")
sys.path.insert(0, str(ROOT))
print("Repository root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display
from torch.utils.data import DataLoader

import segmentation_models_pytorch as smp

from src import analysis as an
from src import reporting as rp
from src.config import PipelineConfig
from src.data import build_datasets, build_file_maps, load_split, load_stats
from src.model import INIT_MEAN, INIT_RGB_ZERO, build_model
from src.pipeline import run_pipeline
from src.preprocessing import CHANNEL_NAMES

print("torch", torch.__version__, "| smp", smp.__version__, "| CUDA:", torch.cuda.is_available())

# ---- edit these two paths if you are not on Kaggle -------------------------------------------
KAGGLE_DATA = Path("/kaggle/input/datasets/nebalelshobary/satellite-multispectral-water-segmentation")
DATA_ROOT = KAGGLE_DATA if KAGGLE_DATA.exists() else Path("data")      # folder with images/ and labels/
OUT_DIR = Path("/kaggle/working/week2_outputs") if Path("/kaggle").exists() else ROOT

# ---- optional Week 1 extras (see README) -----------------------------------------------------
WEEK1_TEST_CSV = None        # CSV with Week 1 test metrics per seed -> enables the test-set comparison
WEEK1_PARAMS_M = None        # Week 1 model size in millions of parameters, e.g. 7.8

cfg = PipelineConfig(data_root=DATA_ROOT, config_dir=ROOT / "config", out_dir=OUT_DIR,
                     readme_path=ROOT / "README.md",
                     week1_test_csv=WEEK1_TEST_CSV, week1_params_m=WEEK1_PARAMS_M)
print(cfg.to_dict())

## 2. Dataset and 12-channel preprocessing

Each sample is a 12-band `128×128` GeoTIFF with a binary water mask. Only bands 0–6 are true spectral bands;
the remaining ones are QA, two DEMs, ESA WorldCover and Water Occurrence (see the Week 1 notebook for the audit of
possible shortcut features). Every band is normalised with statistics computed on the **training split only**.

In [ ]:
image_map, mask_map = build_file_maps(cfg.images_dir, cfg.masks_dir)
split_df, train_df, val_df, test_df = load_split(cfg.split_path)
stats = load_stats(cfg.stats_path)
datasets = build_datasets(train_df, val_df, test_df, image_map, mask_map, stats)

print(f"image/mask pairs: {len(image_map)} | train/val/test: {len(train_df)}/{len(val_df)}/{len(test_df)}")
display(split_df.groupby(["Coverage_Group", "Split"]).size().unstack(fill_value=0))

x, y, sample_id = datasets["train"][0]
print("sample", sample_id, "| image", tuple(x.shape), "| mask", tuple(y.shape), "| finite:", bool(torch.isfinite(x).all()))
assert tuple(x.shape) == (12, 128, 128)          # required output shape of the preprocessing pipeline
print("Channels:", ", ".join(f"{i}:{n}" for i, n in enumerate(CHANNEL_NAMES)))

## 3. Pretrained backbone research

Candidate ImageNet encoders available in `segmentation-models-pytorch`:

| Backbone | Idea | Trade-off |
|---|---|---|
| ResNet18 / 34 / 50 | residual blocks, skip connections ease optimisation | deeper = more capacity, more memory |
| EfficientNet-B0 / B3 | compound scaling of depth/width/resolution, few parameters | depth-wise convs; more sensitive to small data and BatchNorm statistics |

**Choice: ResNet34** — a good capacity/stability trade-off for only 306 patches of 128×128, and its stride-2 stem plus
four residual stages give the multi-scale features a U-Net decoder needs.

**Decoder: U-Net** (matches the Week 1 baseline, skip connections preserve fine water boundaries).
**DeepLabV3+** (Chen et al., arXiv:1606.00915 — atrous/dilated convolutions and Atrous Spatial Pyramid Pooling
enlarge the receptive field without losing resolution) is a valid alternative and can be tried with
`--decoder deeplabv3plus`; it is not part of the main comparison so that only *one* factor changes versus Week 1.

In [ ]:
rows = []
for name, strength, limit in [
    ("resnet18", "light, fast, stable", "lowest capacity"),
    ("resnet34", "capacity / cost balance (selected)", "heavier than ResNet18"),
    ("resnet50", "bottleneck blocks, high capacity", "more compute and memory"),
    ("efficientnet-b0", "very parameter-efficient", "can be fragile on tiny datasets"),
    ("efficientnet-b3", "stronger EfficientNet variant", "higher compute cost"),
]:
    encoder = smp.encoders.get_encoder(name, in_channels=3, depth=5, weights=None)
    rows.append({"Backbone": name, "Encoder params (M)": sum(p.numel() for p in encoder.parameters()) / 1e6,
                 "Strength": strength, "Limitation": limit})
backbone_df = pd.DataFrame(rows)
display(backbone_df.style.format({"Encoder params (M)": "{:.2f}"}))

## 4. Adapting the first convolution from 3 to 12 channels

ImageNet weights exist only for RGB. The stem conv is replaced by a 12-channel conv in two ways:

* **`mean`** — average the R/G/B filters, repeat over 12 channels, scale by 3/12 (keeps the activation scale).
* **`rgb_zero_extra`** — put the R/G/B filters on the dataset's Red/Green/Blue bands (channels 3/2/1 — the dataset
  order is Coastal, Blue, Green, Red, …, *not* R, G, B) and zero-initialise the other nine channels.

Both are trained under identical conditions and the winner is chosen on **mean validation IoU over all seeds**.

In [ ]:
model_mean = build_model("resnet34", "unet", INIT_MEAN)
model_rgb = build_model("resnet34", "unet", INIT_RGB_ZERO)

print("first conv (mean):", model_mean.encoder.conv1)
w = model_rgb.encoder.conv1.weight
print("rgb_zero_extra non-zero input channels:", (w.abs().sum(dim=(0, 2, 3)) > 0).nonzero().flatten().tolist(), "(expected [1, 2, 3])")
assert model_mean.encoder.conv1.in_channels == 12 and model_rgb.encoder.conv1.in_channels == 12

# forward-pass sanity check on one real validation batch
images, masks, _ = next(iter(DataLoader(datasets["val"], batch_size=8)))
with torch.no_grad():
    logits = model_mean.eval()(images)
print("input", tuple(images.shape), "-> logits", tuple(logits.shape))
assert logits.shape == masks.shape and torch.isfinite(logits).all()
del model_mean, model_rgb, logits

## 5. Fine-tuning and evaluation

`run_pipeline` trains every (init strategy × seed) combination plus the random-init control with:
BCE + Dice loss, AdamW (encoder LR 1e-4, decoder LR 1e-3, weight decay 1e-4), `ReduceLROnPlateau`,
mixed precision, early stopping on validation IoU (patience 15, max 100 epochs) and best-checkpoint restoration.
Finished runs are reused automatically if the session restarts (`resume=True`).
The held-out **test set is evaluated only after** the initialisation strategy has been selected on validation data.

In [ ]:
res = run_pipeline(cfg, show_figures=True)
print("\nSelected first-conv initialisation:", res["selected"])

## 6. Results

In [ ]:
display(Markdown("### Model comparison (mean ± std over seeds)"))
display(res["model_comparison"])
display(Markdown("### Per-seed validation IoU"))
display(res["paired_val_iou"])
if res["paired_test_iou"] is not None:
    display(Markdown("### Per-seed test IoU"))
    display(res["paired_test_iou"])
display(Markdown("### First-layer initialisation strategies"))
display(res["strategy_table"])

In [ ]:
display(Markdown("### Held-out test results, every run"))
display(res["runs_test"])
display(Markdown("### Error analysis by water coverage (selected model, test)"))
display(res["coverage_test"])

## 7. Findings (generated from the numbers)

In [ ]:
display(Markdown(rp.make_findings(res)))
display(Markdown("**Limitations**\n\n" + rp.LIMITATIONS))

## 8. Task checklist

| Week 2 requirement | Where |
|---|---|
| Research pretrained backbones | Section 3 |
| Adapt first conv to 12 channels (averaging / zero-init) | Section 4, `src/model.py` |
| `segmentation-models-pytorch` encoder + U-Net (DeepLabV3+ optional) | `src/model.py` |
| Fine-tune on the water dataset | Section 5, `src/engine.py` |
| Compare with the Week 1 scratch U-Net (IoU, F1, Precision, Recall) | Section 6 |
| Document which model is better and why | Section 7 (+ random-init ablation) |
| Clean, modular code in the repository | `src/`, `scripts/`, `tests/` |